## C1 - WEEK 2

- Adrià Cantarero Carreras (1809773@uab.cat)
- Ivan Martin Campoy (1673246@uab.cat)
- Irene Pumares Benaiges (1810704@uab.cat)
- Lavish Ramchandani (1691238@uab.cat)

In [44]:
#needed libraries

import os
import re
import cv2
import pickle
import numpy as np
from functools import partial

## TASK 1: Implement 3D / 2D and block and hierarchical histograms

In [41]:
#database and qsd1 paths

bbdd_path = "./BBDD"
qsd2_path = "./qsd2_w1"

COLOR_SPACES = {
    "GRAY": cv2.COLOR_BGR2GRAY,
    "RGB":  cv2.COLOR_BGR2RGB,
    "HSV":  cv2.COLOR_BGR2HSV,
    "LAB":  cv2.COLOR_BGR2LAB,
    "YCRCB": cv2.COLOR_BGR2YCrCb,
}

CHANNEL_RANGES = {
    "GRAY":  [(0, 256)],
    "RGB":   [(0, 256)] * 3,
    "HSV":   [(0, 180), (0, 256), (0, 256)],
    "LAB":   [(0, 256)] * 3,
    "YCRCB": [(0, 256)] * 3,
}

def read_image(img_path):
    image = cv2.imread(img_path)
    if image is None:
        raise IOError(f"Could not read {img_path}")   #skipping it would shift every index after it and break the ground truth
    return image

bbdd_descriptors = []
qsd2_descriptors = []

#----------------------------------------------------------------------------------
#sort files
def return_number(string):
    return [int(text) if text.isdigit() else text.lower() for text in re.split(r'(\d+)', string)]

#sort bbdd files
bbdd_files = [f for f in os.listdir(bbdd_path) if f.lower().endswith(".jpg")]
bbdd_files.sort(key=return_number)

#sort qsd2 files
qsd2_files = [f for f in os.listdir(qsd2_path) if f.lower().endswith(".jpg")]
qsd2_files.sort(key=return_number)

#id of an image = number in its file name (bbdd_00007.jpg -> 7)
def image_id(file_name):
    return int(re.findall(r'\d+', file_name)[-1])

bbdd_ids = np.array([image_id(f) for f in bbdd_files])
qsd2_ids = np.array([image_id(f) for f in qsd2_files])
#----------------------------------------------------------------------------------

#load images iteratively bbdd set
bbdd_imgs = []
for file_name in bbdd_files:
    img_path = os.path.join(bbdd_path, file_name)
    image = read_image(img_path)
    bbdd_imgs.append(image)

##load images iteratively qsd2 set
qsd2_imgs = []
for file_name in qsd2_files:
    img_path = os.path.join(qsd2_path, file_name)
    image = read_image(img_path)
    qsd2_imgs.append(image)

In [29]:
def compute_nd_histogram(image,color_space = 'HSV', channels = (1,2), bins= 16, mask = None):
    if mask is not None:
        mask = (mask > 0).astype(np.uint8) * 255 #convert boolean mask to 0-255 just in case

    ranges = [] #ranges for the color space channels
    if color_space in COLOR_SPACES:
        converted_image = cv2.cvtColor(image, COLOR_SPACES[color_space])
        if len(converted_image.shape) == 2:
            raise ValueError("Descriptor needs 2 channels at least")

        for c in channels:
            ranges.extend(CHANNEL_RANGES[color_space][c])

    else:
        raise ValueError("Unknown color space: %s" % color_space)

    if isinstance(bins, int):
        hist_size = [bins for i in channels] #ex 16 -> (16,16)
    else:
        assert len(bins) == len(channels), "Number of bins must match number of channels"
        hist_size = bins
  
    hist = cv2.calcHist([converted_image], list(channels), mask, hist_size, ranges)
    hist = hist.flatten()

    #normalize only once
    total = np.sum(hist)
    if total > 0:
        hist = hist / total
    return hist


image = cv2.imread("./BBDD/bbdd_00000.jpg")
d = compute_nd_histogram(image)
print(d.shape, d.sum())


(256,) 1.0


In [34]:
def compute_nd_histogram(image,color_space = 'HSV', channels = (1,2), bins= 16, mask = None):
    if mask is not None:
        mask = (mask > 0).astype(np.uint8) * 255 #convert boolean mask to 0-255 just in case

    ranges = [] #ranges for the color space channels
    if color_space in COLOR_SPACES:
        converted_image = cv2.cvtColor(image, COLOR_SPACES[color_space])
        if len(converted_image.shape) == 2:
            raise ValueError("Descriptor needs 2 channels at least")

        for c in channels:
            ranges.extend(CHANNEL_RANGES[color_space][c])

    else:
        raise ValueError("Unknown color space: %s" % color_space)

    if isinstance(bins, int):
        hist_size = [bins for i in channels] #ex 16 -> (16,16)
    else:
        assert len(bins) == len(channels), "Number of bins must match number of channels"
        hist_size = bins
  
    hist = cv2.calcHist([converted_image], list(channels), mask, hist_size, ranges)
    hist = hist.flatten()

    #normalize only once
    total = np.sum(hist)
    if total > 0:
        hist = hist / total
    return hist


image = cv2.imread("./BBDD/bbdd_00000.jpg")
d = compute_nd_histogram(image)
print(d.shape, d.sum())


(256,) 1.0


In [33]:
def compute_block_histogram(image, grid=(4, 4), color_space='HSV', channels=(1, 2), bins=16, mask=None):
    # 1. block boundaries
    rows, cols = grid
    height, width = image.shape[:2]
    y_cuts  = np.linspace(0, height, rows + 1, dtype=int)
    x_cuts  = np.linspace(0, width, cols + 1, dtype=int)
    block_histograms = []

    # 2. loop over blocks: slice image (and mask), call compute_nd_histogram
    for r in range(rows):
        for c in range(cols):
            y_start, y_end = y_cuts[r], y_cuts[r+1]
            x_start, x_end = x_cuts[c], x_cuts[c+1]
            block_img = image[y_start:y_end, x_start:x_end]
            block_mask = None
            if mask is not None:
                block_mask = mask[y_start:y_end, x_start:x_end]
        
            hist = compute_nd_histogram(block_img, color_space, channels, bins, block_mask)
            block_histograms.append(hist)

    # 3. concatenate (and decide the normalization)
    descriptor = np.concatenate(block_histograms)
    total = np.sum(descriptor)
    if total > 0:
        descriptor = descriptor / total
    return descriptor

image = cv2.imread("./BBDD/bbdd_00000.jpg")
d = compute_block_histogram(image)
print(d.shape, d.sum())

(4096,) 1.0


In [ ]:
def compute_descriptors(images, descriptor_fn):
    # images: list of images (BGR arrays)
    # descriptor_fn: function that takes ONE image and returns a 1D vector

    # 1. container to store one vector per image
    descriptors = ...

    # 2. loop over the images
    for image in images:
        # compute the descriptor of this image and store it
        ...

    # 3. turn the list into a matrix of shape (N, D)
    return ...

## Task2- Test query system using query set QSD1-W2 development, evaluate retrievalresults (use your best performing descriptor). 
Compare results against the best descriptor/distance in W1 (the dev set is the same)

In [39]:
eps = 1e-10  # small value to avoid division by zero


def euclidean_distance(x, y):
    return np.sqrt(np.sum((x - y) ** 2, axis=-1))

def L1_distance(x, y):
    return np.sum(np.abs(x - y), axis=-1)

def chi_squared_distance(x, y):
    return 0.5 * np.sum(((x - y) ** 2) / (x + y + eps), axis=-1)

def jensen_shannon_distance(x, y):
    m = 0.5 * (x + y)
    kl_x = np.sum(x * np.log((x + eps) / (m + eps)), axis=-1)
    kl_y = np.sum(y * np.log((y + eps) / (m + eps)), axis=-1)
    return np.sqrt(np.clip(0.5 * (kl_x + kl_y), 0, None))

#all the measures below are written as distances (smaller = more similar) because the retrieval step will rank every measure 
#with the same rule (sort ascending and take the first k)

def histogram_intersection_distance(x, y):
    return 1.0 - np.sum(np.minimum(x, y), axis=-1)

def hellinger_distance(x, y):
    return (1.0 / np.sqrt(2.0)) * np.sqrt(np.sum((np.sqrt(np.clip(x, 0, None)) - np.sqrt(np.clip(y, 0, None))) ** 2, axis=-1))

def cosine_distance(x, y):
    return 1.0 - np.sum(x * y, axis=-1) / (np.linalg.norm(x, axis=-1) * np.linalg.norm(y, axis=-1) + eps)

def correlation_distance(x, y):
    xc = x - np.mean(x, axis=-1, keepdims=True)
    yc = y - np.mean(y, axis=-1, keepdims=True)
    return 1.0 - np.sum(xc * yc, axis=-1) / (np.linalg.norm(xc, axis=-1) * np.linalg.norm(yc, axis=-1) + eps)

MEASURES = {
    "euclidean":      euclidean_distance,
    "L1":             L1_distance,
    "chi2":           chi_squared_distance,
    "intersection":   histogram_intersection_distance,
    "hellinger":      hellinger_distance,
    "cosine":         cosine_distance,
    "correlation":    correlation_distance,
    "jensen_shannon": jensen_shannon_distance,
}

# import ml_metrics as metrics #Couldn't install it so I copied the code from the git

import numpy as np

#code from https://github.com/benhamner/Metrics/blob/master/Python/ml_metrics/custom/kdd_average_precision.py
def kdd_apk(actual, predicted, k=10):
    """
    Computes the average precision at k for Track 1 of the 2012 KDD Cup.

    This modified version uses the number of actual clicks as the denominator,
    regardless of k.

    This function computes the average prescision at k between two lists of
    items.

    Parameters
    ----------
    actual : list
             A list of elements that are to be predicted (order doesn't matter)
    predicted : list
                A list of predicted elements (order does matter)
    k : int, optional
        The maximum number of predicted elements

    Returns
    -------
    score : double
            The average precision at k over the input lists

    """
    if len(predicted)>k:
        predicted = predicted[:k]

    score = 0.0
    num_hits = 0.0

    for i,p in enumerate(predicted):
        if p in actual and p not in predicted[:i]: #p not in predicted[:i] is to avoid no extra marks for repeating
            num_hits += 1.0
            score += num_hits / (i+1.0)

    if not actual:
        return 0.0

    return score / len(actual)

def kdd_mapk(actual, predicted, k=10):
    """
    Computes the mean average precision at k for Track 1 of the 2012 KDD Cup.

    This modified version uses the number of actual clicks as the denominator,
    regardless of k.

    This function computes the mean average prescision at k between two lists
    of lists of items.

    Parameters
    ----------
    actual : list
             A list of lists of elements that are to be predicted 
             (order doesn't matter in the lists)
    predicted : list
                A list of lists of predicted elements
                (order matters in the lists)
    k : int, optional
        The maximum number of predicted elements

    Returns
    -------
    score : double
            The mean average precision at k over the input lists

    """
    return np.mean([kdd_apk(a,p,k) for a,p in zip(actual, predicted)])


#apk is like the average precision of hits at k items
#but mapk looks in the entire dataset